In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load from organized bangkok folder
DATA_PATH = '../data/bangkok/'

listings = pd.read_csv(DATA_PATH + 'listings.csv')
calendar = pd.read_csv(DATA_PATH + 'calendar.csv')
reviews = pd.read_csv(DATA_PATH + 'reviews.csv')
neighbourhoods = pd.read_csv(DATA_PATH + 'neighbourhoods.csv')

print("All datasets loaded from data/bangkok/")
print(f"listings:       {listings.shape}")
print(f"calendar:       {calendar.shape}")
print(f"reviews:        {reviews.shape}")
print(f"neighbourhoods: {neighbourhoods.shape}")

In [ ]:
# ── 3.1 PROFILING ─────────────────────────────────────────
print("=" * 60)
print("DATASET PROFILING REPORT")
print("=" * 60)

def profile_dataset(df, name):
    print(f"\n{'─' * 60}")
    print(f"DATASET: {name}")
    print(f"{'─' * 60}")
    print(f"Rows:          {df.shape[0]:,}")
    print(f"Columns:       {df.shape[1]}")
    print(f"Memory usage:  {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nData Types:")
    print(df.dtypes.value_counts())
    print(f"\nColumn Cardinality (unique values):")
    cardinality = df.nunique().sort_values(ascending=False)
    print(cardinality.head(10))

profile_dataset(listings, "Listings")
profile_dataset(calendar, "Calendar")
profile_dataset(reviews, "Reviews")
profile_dataset(neighbourhoods, "Neighbourhoods")

In [ ]:
# ── DUPLICATE DETECTION ────────────────────────────────────
print("=" * 60)
print("DUPLICATE DETECTION")
print("=" * 60)

# 1. Exact duplicates
print("\n1. EXACT DUPLICATE ROWS:")
print(f"   Listings:       {listings.duplicated().sum()}")
print(f"   Calendar:       {calendar.duplicated().sum()}")
print(f"   Reviews:        {reviews.duplicated().sum()}")
print(f"   Neighbourhoods: {neighbourhoods.duplicated().sum()}")

# 2. Duplicate IDs
print("\n2. DUPLICATE IDs:")
print(f"   Duplicate listing IDs:  {listings['id'].duplicated().sum()}")
print(f"   Duplicate review IDs:   {reviews['id'].duplicated().sum()}")

# 3. Fuzzy duplicate detection on listings
# Same host, same location, similar price
print("\n3. POTENTIAL FUZZY DUPLICATES IN LISTINGS:")
print("   (Same host_id + neighbourhood + room_type + price)")
fuzzy_dupes = listings.duplicated(
    subset=['host_id', 'neighbourhood_cleansed', 'room_type', 'price'],
    keep=False
)
print(f"   Potential fuzzy duplicates: {fuzzy_dupes.sum()}")
print("\n   Sample fuzzy duplicates:")
print(listings[fuzzy_dupes][['id','host_id','neighbourhood_cleansed',
                              'room_type','price','name']].head(6))

# 4. Duplicate reviews - same listing, same reviewer
print("\n4. DUPLICATE REVIEWS (same listing + same reviewer):")
review_dupes = reviews.duplicated(subset=['listing_id', 'reviewer_id'], keep=False)
print(f"   Duplicate review pairs: {review_dupes.sum()}")

In [ ]:
# ── OUTLIER DETECTION ──────────────────────────────────────
print("=" * 60)
print("OUTLIER DETECTION - KEY NUMERICAL FIELDS")
print("=" * 60)

# First clean price for analysis
listings['price_numeric'] = (
    listings['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

def detect_outliers(series, name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"\n{name}:")
    print(f"   Min:        {series.min():,.2f}")
    print(f"   Max:        {series.max():,.2f}")
    print(f"   Mean:       {series.mean():,.2f}")
    print(f"   Median:     {series.median():,.2f}")
    print(f"   Q1:         {Q1:,.2f}")
    print(f"   Q3:         {Q3:,.2f}")
    print(f"   IQR lower:  {lower:,.2f}")
    print(f"   IQR upper:  {upper:,.2f}")
    print(f"   Outliers:   {len(outliers):,} ({len(outliers)/len(series)*100:.1f}%)")

detect_outliers(listings['price_numeric'].dropna(), "PRICE (THB)")
detect_outliers(listings['availability_365'].dropna(), "AVAILABILITY (days/year)")
detect_outliers(listings['number_of_reviews'].dropna(), "NUMBER OF REVIEWS")
detect_outliers(listings['minimum_nights'].dropna(), "MINIMUM NIGHTS")

In [ ]:
# ── DOMAIN VALIDATION ──────────────────────────────────────
print("=" * 60)
print("DOMAIN VALIDATION")
print("=" * 60)

# 1. Price cannot be negative or zero
invalid_price = listings[listings['price_numeric'] <= 0]
print(f"\n1. Listings with price <= 0: {len(invalid_price)}")

# 2. Price suspiciously low (less than 100 THB)
low_price = listings[listings['price_numeric'] < 100]
print(f"2. Listings with price < 100 THB: {len(low_price)}")
print(low_price[['id','name','price','room_type']].head(5))

# 3. Latitude must be valid for Bangkok (13.5 to 14.0)
invalid_lat = listings[
    (listings['latitude'] < 13.5) | (listings['latitude'] > 14.0)
]
print(f"\n3. Listings with invalid latitude for Bangkok: {len(invalid_lat)}")

# 4. Longitude must be valid for Bangkok (100.3 to 100.9)
invalid_lon = listings[
    (listings['longitude'] < 100.3) | (listings['longitude'] > 100.9)
]
print(f"4. Listings with invalid longitude for Bangkok: {len(invalid_lon)}")

# 5. Availability must be 0-365
invalid_avail = listings[
    (listings['availability_365'] < 0) | (listings['availability_365'] > 365)
]
print(f"\n5. Listings with invalid availability: {len(invalid_avail)}")

# 6. Minimum nights must be positive
invalid_min = listings[listings['minimum_nights'] <= 0]
print(f"6. Listings with minimum_nights <= 0: {len(invalid_min)}")

# 7. Review scores must be between 0 and 5
invalid_scores = listings[
    (listings['review_scores_rating'] < 0) | 
    (listings['review_scores_rating'] > 5)
]
print(f"\n7. Listings with invalid review scores: {len(invalid_scores)}")

# 8. Calendar availability must be t or f
invalid_avail_cal = calendar[~calendar['available'].isin(['t', 'f'])]
print(f"\n8. Calendar rows with invalid availability value: {len(invalid_avail_cal)}")

print("\nDomain validation complete!")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 3.2 - DATA CLEANING & STANDARDIZATION
# ══════════════════════════════════════════════════════════

print("Starting Data Cleaning Pipeline...")
print("=" * 60)

# Work on a copy to preserve original data
listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()

print("Working copies created.")
print(f"listings_clean:  {listings_clean.shape}")
print(f"calendar_clean:  {calendar_clean.shape}")
print(f"reviews_clean:   {reviews_clean.shape}")

In [ ]:
# ── STEP 1: Clean Price Column ─────────────────────────────
print("=" * 60)
print("STEP 1: PRICE CLEANING")
print("=" * 60)

# Convert price from '$1,500.00' format to float
listings_clean['price_numeric'] = (
    listings_clean['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Flag suspicious prices
listings_clean['price_flag'] = 'ok'
listings_clean.loc[listings_clean['price_numeric'] < 100, 'price_flag'] = 'suspiciously_low'
listings_clean.loc[listings_clean['price_numeric'] > 50000, 'price_flag'] = 'suspiciously_high'
listings_clean.loc[listings_clean['price_numeric'].isna(), 'price_flag'] = 'missing'

print(f"Price cleaning complete.")
print(f"\nPrice flag distribution:")
print(listings_clean['price_flag'].value_counts())
print(f"\nPrice stats after cleaning:")
print(listings_clean['price_numeric'].describe())

In [ ]:
# ── STEP 2: Parse Date Fields ──────────────────────────────
print("=" * 60)
print("STEP 2: DATE STANDARDIZATION")
print("=" * 60)

# Listings dates
listings_clean['host_since'] = pd.to_datetime(listings_clean['host_since'], errors='coerce')
listings_clean['first_review'] = pd.to_datetime(listings_clean['first_review'], errors='coerce')
listings_clean['last_review'] = pd.to_datetime(listings_clean['last_review'], errors='coerce')
listings_clean['last_scraped'] = pd.to_datetime(listings_clean['last_scraped'], errors='coerce')

# Calendar dates
calendar_clean['date'] = pd.to_datetime(calendar_clean['date'], errors='coerce')

# Reviews dates
reviews_clean['date'] = pd.to_datetime(reviews_clean['date'], errors='coerce')

print("Date columns converted:")
print(f"  listings - host_since:   {listings_clean['host_since'].dtype}")
print(f"  listings - first_review: {listings_clean['first_review'].dtype}")
print(f"  listings - last_review:  {listings_clean['last_review'].dtype}")
print(f"  calendar - date:         {calendar_clean['date'].dtype}")
print(f"  reviews  - date:         {reviews_clean['date'].dtype}")

print(f"\nDate ranges:")
print(f"  host_since:   {listings_clean['host_since'].min()} to {listings_clean['host_since'].max()}")
print(f"  first_review: {listings_clean['first_review'].min()} to {listings_clean['first_review'].max()}")
print(f"  calendar:     {calendar_clean['date'].min()} to {calendar_clean['date'].max()}")
print(f"  reviews:      {reviews_clean['date'].min()} to {reviews_clean['date'].max()}")

In [ ]:
# ── STEP 3: Normalize Categorical Fields ───────────────────
print("=" * 60)
print("STEP 3: CATEGORICAL NORMALIZATION")
print("=" * 60)

# Room type - already clean but let's standardize
print("Room types before:")
print(listings_clean['room_type'].value_counts())

# Property type - normalize to broader categories
print(f"\nProperty types (top 15):")
print(listings_clean['property_type'].value_counts().head(15))

# Create simplified property category
def simplify_property_type(prop):
    if pd.isna(prop):
        return 'Unknown'
    prop = prop.lower()
    if 'entire' in prop or 'whole' in prop:
        return 'Entire Place'
    elif 'private' in prop:
        return 'Private Room'
    elif 'shared' in prop:
        return 'Shared Room'
    elif 'hotel' in prop or 'hostel' in prop:
        return 'Hotel/Hostel'
    elif 'villa' in prop:
        return 'Villa'
    elif 'condo' in prop or 'apartment' in prop or 'flat' in prop:
        return 'Apartment/Condo'
    elif 'house' in prop or 'home' in prop:
        return 'House'
    else:
        return 'Other'

listings_clean['property_category'] = listings_clean['property_type'].apply(simplify_property_type)

print(f"\nProperty categories after normalization:")
print(listings_clean['property_category'].value_counts())

# Convert available column in calendar from t/f to boolean
calendar_clean['is_available'] = calendar_clean['available'] == 't'
print(f"\nCalendar availability converted to boolean:")
print(calendar_clean['is_available'].value_counts())

In [ ]:
# ── STEP 4: Handle Missing Values ─────────────────────────
print("=" * 60)
print("STEP 4: MISSING VALUE HANDLING")
print("=" * 60)

# 1. neighbourhood - use neighbourhood_cleansed (more reliable)
print(f"neighbourhood missing:           {listings_clean['neighbourhood'].isna().sum()}")
print(f"neighbourhood_cleansed missing:  {listings_clean['neighbourhood_cleansed'].isna().sum()}")

# Use neighbourhood_cleansed as primary neighbourhood field
listings_clean['neighbourhood_standard'] = listings_clean['neighbourhood_cleansed']

# 2. Review scores - fill with -1 as sentinel for "no reviews yet"
review_score_cols = [
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value'
]
for col in review_score_cols:
    listings_clean[col] = listings_clean[col].fillna(-1)

print(f"\nReview scores filled with -1 (sentinel for no reviews)")
print(f"review_scores_rating sample:")
print(listings_clean['review_scores_rating'].value_counts().head(5))

# 3. beds/bedrooms/bathrooms - fill with median
for col in ['beds', 'bedrooms']:
    median_val = listings_clean[col].median()
    listings_clean[col] = listings_clean[col].fillna(median_val)
    print(f"\n{col} - filled {listings[col].isna().sum()} nulls with median ({median_val})")

# 4. host fields - fill with Unknown
listings_clean['host_name'] = listings_clean['host_name'].fillna('Unknown')
listings_clean['host_is_superhost'] = listings_clean['host_is_superhost'].fillna('f')
listings_clean['host_response_rate'] = listings_clean['host_response_rate'].fillna('Unknown')
listings_clean['host_response_time'] = listings_clean['host_response_time'].fillna('Unknown')

# 5. price - keep NaN but add flag (already done in step 1)
print(f"\nMissing prices kept as NaN (flagged in price_flag column)")

# 6. Reviews - drop rows with null comments
reviews_before = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=['comments'])
print(f"\nReviews: dropped {reviews_before - len(reviews_clean)} rows with null comments")
print(f"Reviews remaining: {len(reviews_clean):,}")

In [ ]:
# ── STEP 5: Geographic Standardization ────────────────────
print("=" * 60)
print("STEP 5: GEOGRAPHIC STANDARDIZATION")
print("=" * 60)

# 1. Round coordinates to 5 decimal places (sufficient precision)
listings_clean['latitude'] = listings_clean['latitude'].round(5)
listings_clean['longitude'] = listings_clean['longitude'].round(5)
print("Coordinates rounded to 5 decimal places")

# 2. Flag invalid coordinates for Bangkok
listings_clean['coord_flag'] = 'ok'
invalid_lon_mask = (
    (listings_clean['longitude'] < 100.3) | 
    (listings_clean['longitude'] > 100.9)
)
invalid_lat_mask = (
    (listings_clean['latitude'] < 13.5) | 
    (listings_clean['latitude'] > 14.0)
)
listings_clean.loc[invalid_lon_mask, 'coord_flag'] = 'invalid_longitude'
listings_clean.loc[invalid_lat_mask, 'coord_flag'] = 'invalid_latitude'

print(f"\nCoordinate flags:")
print(listings_clean['coord_flag'].value_counts())
print(f"\nInvalid coordinate listings:")
print(listings_clean[listings_clean['coord_flag'] != 'ok'][
    ['id', 'name', 'latitude', 'longitude', 'neighbourhood_standard', 'coord_flag']
])

# 3. Standardize neighbourhood names - strip whitespace, title case
listings_clean['neighbourhood_standard'] = (
    listings_clean['neighbourhood_standard']
    .str.strip()
    .str.title()
)

print(f"\nNeighbourhood sample after standardization:")
print(listings_clean['neighbourhood_standard'].value_counts().head(10))

# 4. Drop 100% empty columns
cols_to_drop = ['license', 'calendar_updated', 'neighbourhood_group_cleansed']
listings_clean = listings_clean.drop(columns=cols_to_drop)
print(f"\nDropped 100% empty columns: {cols_to_drop}")
print(f"listings_clean shape: {listings_clean.shape}")

In [ ]:
# ── SAVE CLEANED DATASETS ──────────────────────────────────
print("=" * 60)
print("SAVING CLEANED DATASETS")
print("=" * 60)

import os
clean_dir = '../data/bangkok/cleaned/'
os.makedirs(clean_dir, exist_ok=True)

# Save cleaned datasets
listings_clean.to_csv(clean_dir + 'listings_clean.csv', index=False, encoding='utf-8')
calendar_clean.to_csv(clean_dir + 'calendar_clean.csv', index=False, encoding='utf-8')
reviews_clean.to_csv(clean_dir + 'reviews_clean.csv', index=False, encoding='utf-8')

print(f"listings_clean saved:  {listings_clean.shape}")
print(f"calendar_clean saved:  {calendar_clean.shape}")
print(f"reviews_clean saved:   {reviews_clean.shape}")

# Summary of cleaning decisions
print("\n" + "=" * 60)
print("CLEANING DECISIONS LOG")
print("=" * 60)
print("""
1. PRICE: Removed $ and , symbols, cast to float
          Flagged prices <100 THB as suspicious (3 listings)
          Flagged prices >50,000 THB as suspicious (37 listings)
          Missing prices kept as NaN with flag

2. DATES: Converted host_since, first_review, last_review,
          calendar.date, reviews.date to datetime64

3. CATEGORIES: Simplified 50+ property types into 7 categories
               Converted calendar available t/f to boolean

4. MISSING:   neighbourhood_cleansed used as primary (0 nulls)
              Review scores filled with -1 (sentinel for no reviews)
              beds/bedrooms filled with median (1.0)
              host fields filled with Unknown
              70 reviews with null comments dropped

5. GEOGRAPHIC: Coordinates rounded to 5 decimal places
               4 listings flagged for invalid longitude (Nong Chok)
               Neighbourhood names standardized to title case

6. COLUMNS:   Dropped 3 columns that were 100% null
              (license, calendar_updated, neighbourhood_group_cleansed)
""")
print("Section 3.2 - Data Cleaning COMPLETE!")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 3.3 - DATA ENRICHMENT & JOINING
# ══════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1: JOIN LISTINGS WITH REVIEW AGGREGATES")
print("=" * 60)

# Compute review aggregates per listing
review_agg = reviews_clean.groupby('listing_id').agg(
    total_reviews        = ('id', 'count'),
    first_review_date    = ('date', 'min'),
    last_review_date     = ('date', 'max'),
    unique_reviewers     = ('reviewer_id', 'nunique')
).reset_index()

review_agg.columns.name = None
print(f"Review aggregates computed: {review_agg.shape}")
print(review_agg.head(3))

# Join with listings
listings_master = listings_clean.merge(
    review_agg,
    left_on='id',
    right_on='listing_id',
    how='left'
)

# Fill listings with no reviews
listings_master['total_reviews'] = listings_master['total_reviews'].fillna(0)
listings_master['unique_reviewers'] = listings_master['unique_reviewers'].fillna(0)

print(f"\nAfter joining reviews:")
print(f"listings_master shape: {listings_master.shape}")
print(f"Listings with reviews: {(listings_master['total_reviews'] > 0).sum():,}")
print(f"Listings without reviews: {(listings_master['total_reviews'] == 0).sum():,}")

In [ ]:
# ── STEP 2: Integrate Calendar Data ───────────────────────
print("=" * 60)
print("STEP 2: OCCUPANCY & REVENUE FROM CALENDAR")
print("=" * 60)

# Compute occupancy per listing
calendar_agg = calendar_clean.groupby('listing_id').agg(
    total_days        = ('date', 'count'),
    available_days    = ('is_available', 'sum'),
).reset_index()

# Booked days = total days - available days
calendar_agg['booked_days'] = calendar_agg['total_days'] - calendar_agg['available_days']

# Occupancy rate = booked days / total days
calendar_agg['occupancy_rate'] = (
    calendar_agg['booked_days'] / calendar_agg['total_days'] * 100
).round(2)

print(f"Calendar aggregates computed: {calendar_agg.shape}")
print(calendar_agg.head(3))

# Join with listings_master
listings_master = listings_master.merge(
    calendar_agg,
    left_on='id',
    right_on='listing_id',
    how='left',
    suffixes=('', '_cal')
)

# Estimate annual revenue = price * booked_days
listings_master['estimated_annual_revenue'] = (
    listings_master['price_numeric'] * listings_master['booked_days']
).round(2)

print(f"\nAfter joining calendar:")
print(f"listings_master shape: {listings_master.shape}")
print(f"\nOccupancy rate stats:")
print(listings_master['occupancy_rate'].describe())
print(f"\nEstimated annual revenue stats (THB):")
print(listings_master['estimated_annual_revenue'].describe())

In [ ]:
# ── STEP 3: Neighbourhood Aggregates ──────────────────────
print("=" * 60)
print("STEP 3: NEIGHBOURHOOD-LEVEL AGGREGATES")
print("=" * 60)

neighbourhood_agg = listings_master.groupby('neighbourhood_standard').agg(
    listing_count        = ('id', 'count'),
    median_price         = ('price_numeric', 'median'),
    avg_price            = ('price_numeric', 'mean'),
    avg_rating           = ('review_scores_rating', lambda x: x[x > 0].mean()),
    avg_occupancy        = ('occupancy_rate', 'mean'),
    superhost_count      = ('host_is_superhost', lambda x: (x == 't').sum()),
    total_revenue        = ('estimated_annual_revenue', 'sum')
).reset_index()

neighbourhood_agg['superhost_pct'] = (
    neighbourhood_agg['superhost_count'] / 
    neighbourhood_agg['listing_count'] * 100
).round(2)

neighbourhood_agg = neighbourhood_agg.round(2)

print(f"Neighbourhood aggregates: {neighbourhood_agg.shape}")
print("\nTop 10 neighbourhoods by listing count:")
print(neighbourhood_agg.sort_values('listing_count', ascending=False).head(10).to_string())

# Join back to listings_master
listings_master = listings_master.merge(
    neighbourhood_agg[['neighbourhood_standard', 'listing_count', 
                        'median_price', 'avg_occupancy', 'superhost_pct']],
    on='neighbourhood_standard',
    how='left',
    suffixes=('', '_neighbourhood')
)

print(f"\nlistings_master shape after neighbourhood join: {listings_master.shape}")

In [ ]:
# ── STEP 4: Calculated Fields ──────────────────────────────
print("=" * 60)
print("STEP 4: DERIVED/CALCULATED FIELDS")
print("=" * 60)

# Reference date (scrape date)
scrape_date = pd.Timestamp('2025-09-27')

# 1. Host tenure (years on platform)
listings_master['host_tenure_years'] = (
    (scrape_date - listings_master['host_since']).dt.days / 365.25
).round(2)

print("1. Host tenure (years):")
print(listings_master['host_tenure_years'].describe())

# 2. Review frequency (reviews per month since first review)
listings_master['review_frequency_per_month'] = (
    listings_master['total_reviews'] /
    ((scrape_date - listings_master['first_review_date']).dt.days / 30.44)
).round(3)

print("\n2. Review frequency (per month):")
print(listings_master['review_frequency_per_month'].describe())

# 3. Price per bedroom
listings_master['price_per_bedroom'] = (
    listings_master['price_numeric'] / 
    listings_master['bedrooms'].replace(0, 1)
).round(2)

print("\n3. Price per bedroom (THB):")
print(listings_master['price_per_bedroom'].describe())

# 4. Is commercial host (more than 1 listing)
listings_master['is_commercial_host'] = (
    listings_master['calculated_host_listings_count'] > 1
)
print(f"\n4. Commercial hosts (>1 listing):")
print(listings_master['is_commercial_host'].value_counts())

# 5. Listing age in days since first review
listings_master['days_since_last_review'] = (
    scrape_date - listings_master['last_review_date']
).dt.days

print(f"\n5. Days since last review:")
print(listings_master['days_since_last_review'].describe())

In [ ]:
# Fix inf values in review_frequency_per_month
listings_master['review_frequency_per_month'] = (
    listings_master['review_frequency_per_month']
    .replace([float('inf'), float('-inf')], float('nan'))
)

print("Fixed inf values in review_frequency_per_month")
print(listings_master['review_frequency_per_month'].describe())

In [ ]:
# ── SAVE MASTER TABLE ──────────────────────────────────────
print("=" * 60)
print("SAVING LISTINGS MASTER TABLE")
print("=" * 60)

# Select most important columns for master table
master_cols = [
    # Identity
    'id', 'name', 'listing_url',
    # Host
    'host_id', 'host_name', 'host_is_superhost', 
    'host_tenure_years', 'host_response_rate',
    'host_response_time', 'calculated_host_listings_count',
    'is_commercial_host',
    # Location
    'neighbourhood_standard', 'latitude', 'longitude', 'coord_flag',
    # Property
    'room_type', 'property_type', 'property_category',
    'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    # Pricing
    'price', 'price_numeric', 'price_flag', 'price_per_bedroom',
    # Availability
    'minimum_nights', 'maximum_nights', 'availability_365',
    'total_days', 'available_days', 'booked_days', 'occupancy_rate',
    # Revenue
    'estimated_annual_revenue',
    # Reviews
    'number_of_reviews', 'total_reviews', 'unique_reviewers',
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'review_frequency_per_month',
    'days_since_last_review',
    # Dates
    'host_since', 'first_review', 'last_review', 'last_scraped',
    # Neighbourhood aggregates
    'listing_count', 'median_price', 'avg_occupancy', 'superhost_pct'
]

# Keep only columns that exist
master_cols = [c for c in master_cols if c in listings_master.columns]
listings_master_final = listings_master[master_cols].copy()

# Save
clean_dir = '../data/bangkok/cleaned/'
listings_master_final.to_csv(
    clean_dir + 'listings_master.csv', 
    index=False, 
    encoding='utf-8'
)

print(f"listings_master saved!")
print(f"Shape: {listings_master_final.shape}")
print(f"\nColumns included: {len(master_cols)}")
print(f"\nSample of master table:")
print(listings_master_final[['id', 'name', 'neighbourhood_standard', 
                              'price_numeric', 'occupancy_rate',
                              'estimated_annual_revenue',
                              'host_tenure_years']].head(5))

# Save neighbourhood aggregates separately
neighbourhood_agg.to_csv(
    clean_dir + 'neighbourhood_aggregates.csv',
    index=False,
    encoding='utf-8'
)
print(f"\nneighbourhood_aggregates saved!")
print(f"Shape: {neighbourhood_agg.shape}")

print("\nSection 3.3 - Data Enrichment COMPLETE!")

In [ ]:
pip install duckdb

In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 3.4 - DATA MODELING (STAR SCHEMA)
# ══════════════════════════════════════════════════════════
import duckdb

print("=" * 60)
print("STAR SCHEMA DESIGN")
print("=" * 60)
print("""
FACT TABLE:
  fact_listings          ← one row per listing
    listing_id (PK)
    host_id (FK)
    neighbourhood_id (FK)
    property_type_id (FK)
    price_numeric
    occupancy_rate
    booked_days
    estimated_annual_revenue
    total_reviews
    review_scores_rating
    availability_365

DIMENSION TABLES:
  dim_host               ← host details
    host_id (PK)
    host_name
    host_is_superhost
    host_tenure_years
    host_response_rate
    calculated_host_listings_count
    is_commercial_host

  dim_neighbourhood      ← neighbourhood details
    neighbourhood_id (PK)
    neighbourhood_name
    listing_count
    median_price
    avg_occupancy
    superhost_pct

  dim_property           ← property details
    property_type_id (PK)
    room_type
    property_type
    property_category

  dim_date               ← date dimension for calendar
    date_id (PK)
    date
    year, month, day
    day_of_week
    is_weekend
""")

# Connect to DuckDB
con = duckdb.connect('../data/bangkok/airbnb_bangkok.duckdb')
print("DuckDB connected!")

In [ ]:
# ── CREATE DIMENSION TABLES ────────────────────────────────
print("=" * 60)
print("CREATING DIMENSION TABLES")
print("=" * 60)

# 1. dim_host
con.execute("""
    CREATE OR REPLACE TABLE dim_host AS
    SELECT DISTINCT
        host_id,
        host_name,
        host_is_superhost,
        host_tenure_years,
        host_response_rate,
        host_response_time,
        calculated_host_listings_count,
        is_commercial_host
    FROM listings_master_final
""")
print(f"dim_host created: {con.execute('SELECT COUNT(*) FROM dim_host').fetchone()[0]:,} rows")

# 2. dim_neighbourhood
con.execute("""
    CREATE OR REPLACE TABLE dim_neighbourhood AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY neighbourhood_standard) AS neighbourhood_id,
        neighbourhood_standard AS neighbourhood_name,
        listing_count,
        median_price,
        avg_occupancy,
        superhost_pct
    FROM neighbourhood_agg
""")
print(f"dim_neighbourhood created: {con.execute('SELECT COUNT(*) FROM dim_neighbourhood').fetchone()[0]:,} rows")

# 3. dim_property
con.execute("""
    CREATE OR REPLACE TABLE dim_property AS
    SELECT DISTINCT
        ROW_NUMBER() OVER (ORDER BY room_type, property_type) AS property_type_id,
        room_type,
        property_type,
        property_category
    FROM listings_master_final
""")
print(f"dim_property created: {con.execute('SELECT COUNT(*) FROM dim_property').fetchone()[0]:,} rows")

# 4. dim_date
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT DISTINCT
        CAST(STRFTIME(date, '%Y%m%d') AS INTEGER) AS date_id,
        date,
        YEAR(date)          AS year,
        MONTH(date)         AS month,
        DAY(date)           AS day,
        DAYOFWEEK(date)     AS day_of_week,
        DAYNAME(date)       AS day_name,
        MONTHNAME(date)     AS month_name,
        CASE WHEN DAYOFWEEK(date) IN (1,7) THEN true ELSE false END AS is_weekend
    FROM calendar_clean
    WHERE date IS NOT NULL
""")
print(f"dim_date created: {con.execute('SELECT COUNT(*) FROM dim_date').fetchone()[0]:,} rows")

print("\nAll dimension tables created!")

In [ ]:
# Fix dim_property - should have unique combinations only
con.execute("""
    CREATE OR REPLACE TABLE dim_property AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY room_type, property_type) AS property_type_id,
        room_type,
        property_type,
        property_category
    FROM (
        SELECT DISTINCT room_type, property_type, property_category
        FROM listings_master_final
        WHERE property_type IS NOT NULL
    )
""")
count = con.execute('SELECT COUNT(*) FROM dim_property').fetchone()[0]
print(f"dim_property fixed: {count} unique property types")
print(con.execute('SELECT * FROM dim_property LIMIT 10').df())

In [ ]:
# ── CREATE FACT TABLE ──────────────────────────────────────
print("=" * 60)
print("CREATING FACT TABLE")
print("=" * 60)

con.execute("""
    CREATE OR REPLACE TABLE fact_listings AS
    SELECT
        l.id                        AS listing_id,
        l.host_id,
        dn.neighbourhood_id,
        dp.property_type_id,
        l.price_numeric,
        l.price_flag,
        l.occupancy_rate,
        l.booked_days,
        l.available_days,
        l.availability_365,
        l.estimated_annual_revenue,
        l.total_reviews,
        l.unique_reviewers,
        l.review_scores_rating,
        l.review_scores_cleanliness,
        l.review_scores_location,
        l.review_scores_communication,
        l.review_scores_value,
        l.review_frequency_per_month,
        l.days_since_last_review,
        l.minimum_nights,
        l.accommodates,
        l.bedrooms,
        l.beds,
        l.price_per_bedroom,
        l.host_tenure_years,
        l.is_commercial_host,
        l.coord_flag
    FROM listings_master_final l
    LEFT JOIN dim_neighbourhood dn
        ON l.neighbourhood_standard = dn.neighbourhood_name
    LEFT JOIN dim_property dp
        ON l.property_type = dp.property_type
        AND l.room_type = dp.room_type
""")

count = con.execute('SELECT COUNT(*) FROM fact_listings').fetchone()[0]
print(f"fact_listings created: {count:,} rows")
print("\nSample:")
print(con.execute("""
    SELECT listing_id, host_id, neighbourhood_id, 
           property_type_id, price_numeric, occupancy_rate,
           estimated_annual_revenue, total_reviews
    FROM fact_listings 
    LIMIT 5
""").df())

In [ ]:
# ── SQL ANALYTICAL QUERIES ─────────────────────────────────
print("=" * 60)
print("ANALYTICAL SQL QUERIES")
print("=" * 60)

# Query 1: Average price by room type
print("\n1. AVERAGE PRICE BY ROOM TYPE:")
print(con.execute("""
    SELECT 
        dp.room_type,
        COUNT(f.listing_id)              AS total_listings,
        ROUND(AVG(f.price_numeric), 2)   AS avg_price,
        ROUND(MEDIAN(f.price_numeric), 2) AS median_price,
        ROUND(AVG(f.occupancy_rate), 2)  AS avg_occupancy
    FROM fact_listings f
    JOIN dim_property dp ON f.property_type_id = dp.property_type_id
    WHERE f.price_numeric IS NOT NULL
    GROUP BY dp.room_type
    ORDER BY avg_price DESC
""").df().to_string())

# Query 2: Top 10 neighbourhoods by revenue
print("\n2. TOP 10 NEIGHBOURHOODS BY TOTAL REVENUE:")
print(con.execute("""
    SELECT
        dn.neighbourhood_name,
        COUNT(f.listing_id)                       AS listings,
        ROUND(AVG(f.price_numeric), 0)            AS avg_price,
        ROUND(AVG(f.occupancy_rate), 1)           AS avg_occupancy,
        ROUND(SUM(f.estimated_annual_revenue), 0) AS total_revenue
    FROM fact_listings f
    JOIN dim_neighbourhood dn ON f.neighbourhood_id = dn.neighbourhood_id
    WHERE f.estimated_annual_revenue IS NOT NULL
    GROUP BY dn.neighbourhood_name
    ORDER BY total_revenue DESC
    LIMIT 10
""").df().to_string())

# Query 3: Superhost vs non-superhost performance
print("\n3. SUPERHOST VS NON-SUPERHOST PERFORMANCE:")
print(con.execute("""
    SELECT
        h.host_is_superhost,
        COUNT(f.listing_id)                      AS listings,
        ROUND(AVG(f.price_numeric), 2)           AS avg_price,
        ROUND(AVG(f.occupancy_rate), 2)          AS avg_occupancy,
        ROUND(AVG(f.review_scores_rating), 3)    AS avg_rating,
        ROUND(AVG(f.estimated_annual_revenue), 0) AS avg_revenue
    FROM fact_listings f
    JOIN dim_host h ON f.host_id = h.host_id
    WHERE f.price_numeric IS NOT NULL
      AND f.review_scores_rating > 0
    GROUP BY h.host_is_superhost
    ORDER BY h.host_is_superhost
""").df().to_string())

# Query 4: Occupancy by month
print("\n4. CALENDAR AVAILABILITY BY MONTH:")
print(con.execute("""
    SELECT
        d.month_name,
        d.month,
        COUNT(*)                                    AS total_days,
        SUM(CASE WHEN c.available = 'f' THEN 1 ELSE 0 END) AS booked_days,
        ROUND(SUM(CASE WHEN c.available = 'f' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS occupancy_pct
    FROM calendar_clean c
    JOIN dim_date d ON CAST(STRFTIME(c.date, '%Y%m%d') AS INTEGER) = d.date_id
    GROUP BY d.month, d.month_name
    ORDER BY d.month
""").df().to_string())

In [ ]:
# Save schema documentation
con.execute("""
    CREATE OR REPLACE TABLE data_lineage AS
    SELECT * FROM (VALUES
        ('fact_listings',     'listings_master_final', 'Enriched listings with joins'),
        ('dim_host',          'listings_master_final', 'Unique host attributes'),
        ('dim_neighbourhood', 'neighbourhood_agg',     'Neighbourhood aggregates'),
        ('dim_property',      'listings_master_final', 'Unique property types'),
        ('dim_date',          'calendar_clean',        'Date dimension from calendar')
    ) AS t(table_name, source, description)
""")

print("Data lineage table created!")
print(con.execute("SELECT * FROM data_lineage").df().to_string())

# List all tables in DuckDB
print("\nAll tables in DuckDB:")
print(con.execute("SHOW TABLES").df().to_string())

con.close()
print("\nDuckDB connection closed!")
print("Section 3.4 - Data Modeling COMPLETE!")